<a href="https://colab.research.google.com/github/Omar-Gemy/Depi_Graduation_Project-CAI4_AIS3_S4-/blob/translation/seperation%26translation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/depi_project')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
!pip install -q faster-whisper pydub

In [3]:
!apt-get install -q -y ffmpeg

Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [18]:
video_path= '/content/drive/MyDrive/depi_project/input_video2.mp4'

Extractinng audio from video

In [19]:
import os
def get_audio_from_video(video_path):
  audio_path="raw_audio.wav"
  !ffmpeg -i "{video_path}" -ar 16000 -ac 1 -c:a pcm_s16le "{audio_path}" -y -q:a 0
  return audio_path
audio_result = get_audio_from_video("input_video2.mp4")
print(f"Audio is ready here: {audio_result}")



ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

Source Seperating

In [20]:
!pip install uv -q
!pip install torch torchaudio -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 50.4 MB/s eta 0:00:00


In [26]:
!pip install demucs -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 1.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.1/87.1 kB 1.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.3/249.3 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 1.4 MB/s eta 0:00:00


In [27]:
!demucs --two-stems=vocals "/content/drive/MyDrive/depi_project/raw_audio.wav"

Important: the default model was recently changed to `htdemucs` the latest Hybrid Transformer Demucs model. In some cases, this model can actually perform worse than previous models. To get back the old default model use `-n mdx_extra_q`.
Downloading: "https://dl.fbaipublicfiles.com/demucs/hybrid_transformer/955717e8-8726e21a.th" to /root/.cache/torch/hub/checkpoints/955717e8-8726e21a.th
100% 80.2M/80.2M [00:00<00:00, 126MB/s]
Selected model is a bag of 1 models. You will see that many progress bars per track.
Separated tracks will be stored in /content/drive/MyDrive/depi_project/separated/htdemucs
Separating track /content/drive/MyDrive/depi_project/raw_audio.wav
100%|████████████████████████████████████████████████| 99.44999999999999/99.44999999999999 [04:08<00:00,  2.50s/seconds]
/usr/local/lib/python3.12/dist-packages/torchaudio/__init__.py:178: UserWarning: The 'encoding' parameter is not fully supported by TorchCodec AudioEncoder.
  return save_with_torchcodec(
/usr/local/lib/pyt

audio to text

In [34]:
from faster_whisper import WhisperModel

model = WhisperModel("medium", device="cpu", compute_type="int8")

In [36]:
def speech_to_text(audio_file):

    print("Extracting text, please wait...")

    segments, info = model.transcribe(
        audio_file,
        beam_size=5,
        word_timestamps=True
    )

    results = []

    for segment in segments:
        results.append({
            "start": segment.start,
            "end": segment.end,
            "text": segment.text.strip()
        })

        print(f"[{segment.start:.2f}s -> {segment.end:.2f}s] {segment.text}")

    return results, info.language


vocals_path = "/content/drive/MyDrive/depi_project/separated/htdemucs/raw_audio/vocals.wav"

final_text, detected_lang = speech_to_text(vocals_path)

print("Detected Language:", detected_lang)

Extracting text, please wait...
[0.00s -> 3.48s]  Today's networks face thousands of threats every single moment.
[4.18s -> 7.44s]  Now imagine being able to see it all from inside the system itself,
[8.02s -> 11.68s]  watching every bite move, every unusual activity
[11.68s -> 13.48s]  before it becomes an attack.
[14.32s -> 17.58s]  In the real world, human response time isn't always enough.
[18.14s -> 20.24s]  Attacks slip through in fractions of a second,
[20.76s -> 23.52s]  and just one breach can shut down an entire system
[23.52s -> 25.34s]  and cost companies millions.
[25.88s -> 28.70s]  That's why we introduced the AI-powered DevOps system,
[28.70s -> 32.66s]  an intelligent platform that merges the power of artificial intelligence
[32.66s -> 36.94s]  with fully automated DevOps workflows to monitor, analyze,
[37.08s -> 38.72s]  and protect networks in real time.
[39.24s -> 42.14s]  The system begins by collecting live network data
[42.14s -> 45.16s]  using tools like Prometh

Translations

In [37]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import json

In [38]:
model_name = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [39]:
def translate(text, tgt_lang):

    src_lang = "eng_Latn"  # Whisper output = English

    tokenizer.src_lang = src_lang

    inputs = tokenizer(text, return_tensors="pt")

    outputs = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt_lang),
        max_length=400
    )

    return tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

In [42]:
languages = {
    "Arabic": "arb_Arab",
    "French": "fra_Latn",
    "Spanish": "spa_Latn",
    "German": "deu_Latn"
}

Json file with multi-language

In [43]:
def build_multilang_json(whisper_results):

    final_data = []

    for seg in whisper_results:

        text = seg["text"]

        translations = {}

        for lang_name, lang_code in languages.items():
            translations[lang_name] = translate(text, lang_code)

        final_data.append({
            "start": seg["start"],
            "end": seg["end"],
            "text": text,   # English original
            "translations": translations
        })

        print("Processed:", text)

    return final_data

In [44]:
def save_to_json(data, filename="multilang_output.json"):

    with open(filename, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=4)

    print("Saved:", filename)

In [45]:

json_output = build_multilang_json(final_text)

save_to_json(json_output)

Processed: Today's networks face thousands of threats every single moment.
Processed: Now imagine being able to see it all from inside the system itself,
Processed: watching every bite move, every unusual activity
Processed: before it becomes an attack.
Processed: In the real world, human response time isn't always enough.
Processed: Attacks slip through in fractions of a second,
Processed: and just one breach can shut down an entire system
Processed: and cost companies millions.
Processed: That's why we introduced the AI-powered DevOps system,
Processed: an intelligent platform that merges the power of artificial intelligence
Processed: with fully automated DevOps workflows to monitor, analyze,
Processed: and protect networks in real time.
Processed: The system begins by collecting live network data
Processed: using tools like Prometheus and Grafana,
Processed: then applies machine learning models built with TensorFlow
Processed: to understand normal behavior and detect any anomalies,